# Part 6 — Quality check of the aligned data (plan, section 3.6)

**Goal:**
1. QA table and `processed/meta_qa.csv` with one `ok_<name>` flag per embedding;
2. figures: Sentinel-2 RGB, labels and a 3-component PCA of each embedding with the parcel boundaries, for 5 patches per tile, plus the 8 patches whose alignment improves most with a one-pixel shift;
3. a summary of the no-data pixels, to fix how they are handled in training;
4. (optional) the native grid of AlphaEarth in Earth Engine, to explain its extra vertical offset.

**Needs:** Parts 1, 3 and 5 on the Drive; step 6 also needs the Earth Engine project. **Runtime:** CPU. **Time:** about 5 min.

**Files needed in the repository** (new or changed in this part): `scripts/qa_alignment.py` (new flag rule), `scripts/export_rf_layout.py` (docstring fix), `notebooks/data/06_qa.ipynb`.

## 1. Runtime, Drive and configuration
The Earth Engine project (step 6 only) is read from a Colab secret named `GEE_PROJECT` if it exists (key icon in the left bar → Add new secret → name `GEE_PROJECT`, value = the project ID, notebook access on); otherwise edit the line below.

In [ ]:
import sys, shutil, platform
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())

from google.colab import drive
drive.mount("/content/drive")

GEE_PROJECT = "your-project-id"   # <-- EDIT if no Colab secret
try:
    from google.colab import userdata
    GEE_PROJECT = userdata.get("GEE_PROJECT")
except Exception:
    pass
print("Earth Engine project:", GEE_PROJECT)

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

PROC_DRIVE = DRIVE / "processed"
SUBSET = DRIVE / "pastis_subset"
QA_OUT = Path("/content/qa")
NAMES = ["tessera_2019", "tessera_2018", "ae_2019", "ae_2018"]

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

## 3. QA table and figures
The script reads the merged arrays directly on the Drive (only the shown patches are loaded) and writes `meta_qa.csv` next to them.

A patch is **not ok** for an embedding if it has a download error, more than 5% no-data pixels, or a **suspect shift**: moving the embedding by one pixel lowers the within-parcel variance ratio by more than 10%. The sub-pixel offsets measured in Parts 3–5 give much smaller gains and are not flagged.

**Done when:** the table is printed and 5 figures are saved (`qa_t30uxv.png`, `qa_t31tfj.png`, `qa_t31tfm.png`, `qa_t32ulu.png`, `qa_largest_shift_gain.png`).

In [ ]:
!python scripts/qa_alignment.py --pastis "{SUBSET}" --processed "{PROC_DRIVE}" --out "{QA_OUT}"

## 4. Look at the figures
In each row: S2 RGB (clearest summer-2019 date), labels (void in black), then the PCA of each embedding; white lines are the parcel boundaries.

**What to check:** colour changes in the PCA panels follow the white lines; the S2 image matches the labels; TESSERA 2018 and AlphaEarth 2018 may differ inside parcels (crop rotation), not at the boundaries. Patches without S2 in the subset show a black RGB panel. Write down any patch where the PCA is clearly offset from the boundaries.

In [ ]:
from IPython.display import Image, display
for p in sorted(QA_OUT.glob("*.png")):
    print(p.name)
    display(Image(str(p)))

In [ ]:
shutil.copytree(QA_OUT, DRIVE / "results" / "qa", dirs_exist_ok=True)
print(sorted(p.name for p in (DRIVE / "results" / "qa").iterdir()))

## 5. No-data pixels
TESSERA has no-data pixels in some patches; AlphaEarth has none. All variants must be trained and evaluated on the same pixels, so the rule has to be decided once for both methods (e.g. mask these pixels out of the loss and the metrics for every variant).

In [ ]:
rows = []
for name in NAMES:
    nd = pd.read_csv(PROC_DRIVE / f"qa_{name}.csv", index_col="idx").nodata_frac
    rows.append({"name": name, "patches_with_nodata": int((nd > 0).sum()), "max_fraction": round(nd.max(), 4),
                 "nodata_pixels": int((nd * 128 * 128).round().sum()), "share_of_all_pixels": f"{nd.mean():.5%}"})
print(pd.DataFrame(rows).to_string(index=False))

## 6. (Optional) Native grid of AlphaEarth
Reads the projection of one AlphaEarth image per tile. If the origin of the native grid is not a multiple of 10 m (`origin_*_mod10` ≠ 0), the nearest-neighbour request on the snapped PASTIS grid picks a pixel whose centre is offset by that amount, which would explain part of the vertical offset measured in Part 5.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
meta = pd.read_csv(PROC_DRIVE / "meta.csv", index_col="idx")
col = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate("2019-01-01", "2020-01-01")
rows = []
for tile, sub in meta.groupby("TILE"):
    r = sub.iloc[0]
    point = ee.Geometry.Point([(r.lon_min + r.lon_max) / 2, (r.lat_min + r.lat_max) / 2])
    proj = col.filterBounds(point).first().select("A00").projection().getInfo()
    t = proj["transform"]
    rows.append({"tile": tile, "patch_epsg": int(r.epsg), "ae_crs": proj["crs"], "scale_x": t[0], "scale_y": t[4],
                 "origin_x": t[2], "origin_y": t[5], "origin_x_mod10": round(t[2] % 10, 3),
                 "origin_y_mod10": round(t[5] % 10, 3)})
print(pd.DataFrame(rows).to_string(index=False))

## Results to report
1. The commit line of step 2.
2. The table of step 3, and what you saw in the figures (patch IDs that look wrong, if any).
3. The table of step 5.
4. The table of step 6 if it ran.
5. Any error message, in full.

## Checklist G0 (15/11)
| Condition | Evidence |
|---|---|
| PASTIS ready | `processed/meta.csv` (2,433 patches), `labels.npy`, `instances.npy`, `parcel_ids.npy`, `splits.json` |
| TESSERA v1.1 2018 + 2019, same version | `tessera_<year>_info.json`: `1.1-dclimate` |
| AlphaEarth 2018 + 2019 | `ae_<year>.npy`, `ae_<year>_info.json` (`DATASET_VERSION` 1.0, `MODEL_VERSION` 2.1) |
| QA passed | `results/qa/*.png`, `qa_summary.csv`, `meta_qa.csv`, `results/subpixel_offset.csv` |